In [1]:
import torch
from torch import nn, optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, silhouette_score
from sklearn.metrics import davies_bouldin_score
from sklearn.decomposition import PCA
from scipy.stats import chi2
from sklearn.mixture import GaussianMixture
from collections import deque, defaultdict

# CAE

In [2]:
train = pd.read_csv('../../../train_CADE_cod.csv')
val = pd.read_csv('../../../val_CADE_cod.csv')
test = pd.read_csv('../../../test_CADE_cod.csv')

In [3]:
classes_out = ['Infilteration','DoS attacks-SlowHTTPTest']
train = train.drop(index=train[train['Label'].isin(classes_out)].index)
val = val.drop(index=val[val['Label'].isin(classes_out)].index)
test = test.drop(index=test[test['Label'].isin(classes_out)].index)

# CAE CADE (margin 1 e lambda 1.0)

In [4]:
def pick_pair(embeddings, labels, pick_embeddings, pick_labels):
    """
    Gera todos os pares entre embeddings e pick_embeddings,
    com labels_pair indicando se os pares são positivos (0) ou negativos (1).
    """
    N, D = embeddings.shape
    M = pick_embeddings.shape[0]

    # Expande para todas as combinações possíveis
    A = embeddings.unsqueeze(0).expand(M, N, D).reshape(M * N, D)
    B = pick_embeddings.unsqueeze(1).expand(M, N, D).reshape(M * N, D)

    labels_exp = labels.unsqueeze(0).expand(M, N)
    pick_labels_exp = pick_labels.unsqueeze(1).expand(M, N)
    labels_pair = (labels_exp != pick_labels_exp).reshape(-1).float()

    return A, B, labels_pair

def ContrastiveLoss(input1, input2, y, margin=1.0):
    # y = 0 quando input1 e input2 são da mesma classe e y = 1 caso contrário
    diff = input1 - input2
    dist_sq = torch.sum(torch.pow(diff,2),1)
    dist = torch.sqrt(dist_sq + 1e-10) # Soma 1e-10 pois se o sqrt der 0 o gradiente da NaN
    mdist = margin - dist
    mdist = F.relu(mdist)
    loss = ((1-y) * dist * dist) + (y * mdist * mdist)
    return torch.mean(loss)

def AEContrastive(input, output, embeddings, labels, pick_encoded, pick_labels, margin=1.0, lambda1=1.0):
    mseloss = nn.MSELoss()
    ae_loss = mseloss(input,output)

    A, B, labels_pair = pick_pair(embeddings, labels, pick_encoded, pick_labels)
    contrastive_loss = ContrastiveLoss(A, B, labels_pair, margin=margin)
    return ae_loss + contrastive_loss * lambda1, ae_loss, contrastive_loss

In [5]:
class contrastive_ae(nn.Module):
    def __init__(self, input_neurons, neurons):
        super().__init__()
        self.encoder0 = nn.Linear(input_neurons,64)
        self.encoder1 = nn.Linear(64,32)
        self.encoder2 = nn.Linear(32,16)
        self.encoder3 = nn.Linear(16,neurons)

        self.decoder0 = nn.Linear(neurons,16)
        self.decoder1 = nn.Linear(16,32)
        self.decoder2 = nn.Linear(32,64)
        self.decoder3 = nn.Linear(64,input_neurons)

        self.activation0 = nn.ReLU()
    
    def forward(self, X):
        X = self.activation0(self.encoder0(X))
        X = self.activation0(self.encoder1(X))
        X = self.activation0(self.encoder2(X))
        X = self.encoder3(X)
        encoded = X
        X = self.activation0(self.decoder0(X))
        X = self.activation0(self.decoder1(X))
        X = self.activation0(self.decoder2(X))
        X = self.decoder3(X)


        return X, encoded

In [6]:
class encoder(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.encoder0 = list(model.children())[0]
        self.encoder1 = list(model.children())[1]
        self.encoder2 = list(model.children())[2]
        self.encoder3 = list(model.children())[3]
        self.activation0 = list(model.children())[8]
    
    def forward(self, X):
        X = self.activation0(self.encoder0(X))
        X = self.activation0(self.encoder1(X))
        X = self.activation0(self.encoder2(X))
        X = self.encoder3(X)
        return X

In [7]:
from torch.utils.data import Dataset, DataLoader,TensorDataset
import random
def build_class_reference_batches(X_train, y_train, batch_size=512):
    classes = torch.unique(y_train).tolist()
    n_classes = len(classes)
    total_samples = len(X_train)
    dataset = TensorDataset(X_train, y_train)
    train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    

    print(classes)

    # Mapeia cada classe para seus índices
    class_to_indices = {
        int(cls.item()): (y_train == cls).nonzero(as_tuple=True)[0].tolist()
        for cls in torch.unique(y_train)
    }

    reference_list = []
    total_batches = len(train_loader)

    for batch_idx in range(total_batches):
        start = batch_idx * batch_size
        end = min(start + batch_size, total_samples)
        batch_indices = set(range(start, end))

        class_samples = []
        for cls in classes:
            candidates = [i for i in class_to_indices[cls] if i not in batch_indices]
            if not candidates:
                raise ValueError(f"Classe {cls} não possui amostras fora do batch {batch_idx}")
            chosen = candidates[torch.randint(len(candidates), (1,)).item()]
            class_samples.append(chosen)

        reference_list.append(class_samples)


    return train_loader, reference_list

In [8]:
def normalize(t):
    t_norm = (t - t.mean()) / t.std()
    return t_norm


In [9]:
train['Label'] = train['Label'].replace('FTP-BruteForce','BruteForce')
train['Label'] = train['Label'].replace('SSH-Bruteforce','BruteForce')
# train['Label'] = train['Label'].replace('DoS attacks-GoldenEye','DoS')
train['Label'] = train['Label'].replace('DoS attacks-Slowloris','DoS')
## train['Label'] = train['Label'].replace('DoS attacks-SlowHTTPTest','DoS')
train['Label'] = train['Label'].replace('DoS attacks-Hulk','DoS')
train['Label'] = train['Label'].replace('DDoS attacks-LOIC-HTTP','DDoS')
train['Label'] = train['Label'].replace('DDOS attack-LOIC-UDP','DDoS')
train['Label'] = train['Label'].replace('DDOS attack-HOIC','DDoS')
train['Label'] = train['Label'].replace('Brute Force -Web','Web')
train['Label'] = train['Label'].replace('Brute Force -XSS','Web')
train['Label'] = train['Label'].replace('SQL Injection','Web')


val['Label'] = val['Label'].replace('FTP-BruteForce','BruteForce')
val['Label'] = val['Label'].replace('SSH-Bruteforce','BruteForce')
# val['Label'] = val['Label'].replace('DoS attacks-GoldenEye','DoS')
val['Label'] = val['Label'].replace('DoS attacks-Slowloris','DoS')
## val['Label'] = val['Label'].replace('DoS attacks-SlowHTTPTest','DoS')
val['Label'] = val['Label'].replace('DoS attacks-Hulk','DoS')
val['Label'] = val['Label'].replace('DDoS attacks-LOIC-HTTP','DDoS')
val['Label'] = val['Label'].replace('DDOS attack-LOIC-UDP','DDoS')
val['Label'] = val['Label'].replace('DDOS attack-HOIC','DDoS')
val['Label'] = val['Label'].replace('Brute Force -Web','Web')
val['Label'] = val['Label'].replace('Brute Force -XSS','Web')
val['Label'] = val['Label'].replace('SQL Injection','Web')


test['Label'] = test['Label'].replace('FTP-BruteForce','BruteForce')
test['Label'] = test['Label'].replace('SSH-Bruteforce','BruteForce')
# test['Label'] = test['Label'].replace('DoS attacks-GoldenEye','DoS')
test['Label'] = test['Label'].replace('DoS attacks-Slowloris','DoS')
## test['Label'] = test['Label'].replace('DoS attacks-SlowHTTPTest','DoS')
test['Label'] = test['Label'].replace('DoS attacks-Hulk','DoS')
test['Label'] = test['Label'].replace('DDoS attacks-LOIC-HTTP','DDoS')
test['Label'] = test['Label'].replace('DDOS attack-LOIC-UDP','DDoS')
test['Label'] = test['Label'].replace('DDOS attack-HOIC','DDoS')
test['Label'] = test['Label'].replace('Brute Force -Web','Web')
test['Label'] = test['Label'].replace('Brute Force -XSS','Web')
test['Label'] = test['Label'].replace('SQL Injection','Web')

In [10]:
labels_str = [i for i in train['Label'].value_counts().index]
print(labels_str)

for i, l in enumerate(labels_str):
    train['Label'] = train['Label'].replace(l, i)
    val['Label'] = val['Label'].replace(l, i)
    test['Label'] = test['Label'].replace(l, i)

['DDoS', 'Benign', 'DoS', 'BruteForce', 'Bot', 'DoS attacks-GoldenEye', 'Web']


/tmp/ipykernel_887182/2307520911.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train['Label'] = train['Label'].replace(l, i)
/tmp/ipykernel_887182/2307520911.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  val['Label'] = val['Label'].replace(l, i)
/tmp/ipykernel_887182/2307520911.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('f

In [11]:
array_hidden_classes = [[5]]
filenames = ['5']
# array_hidden_classes = [[0]]
# filenames = ['0']
total_classes = len(labels_str)
for i in range(len(array_hidden_classes)):
    filename = f'{filenames[i]}_hidden'
    print(filename)
    hidden_classes = array_hidden_classes[i] #Classes ocultas do treinamento

    hidden_classes_index = list(train[train['Label'].isin(hidden_classes)].index)

    train_hidden = train.loc[hidden_classes_index]

    train_not_hidden = train.drop(hidden_classes_index)

    X_train = train_not_hidden.drop(columns=['Label'])
    y_train = train_not_hidden['Label']
    X_val = val.drop(columns=['Label'])
    y_val = val['Label'].values
    X_test = test.drop(columns=['Label'])
    y_test = test['Label'].values

    torch.manual_seed(123)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(device)
    X_train = torch.tensor(np.array(X_train), dtype=torch.float)
    y_train = torch.tensor(np.array(y_train), dtype=torch.float)

    train_loader, pick_list = build_class_reference_batches(X_train,y_train, batch_size=512)
    pick_samples = []
    pick_labels = []
    for l in pick_list:
        pick_samples.append(X_train[l])
        pick_labels.append(y_train[l])

    pick_samples = torch.stack(pick_samples)    
    pick_labels = torch.stack(pick_labels)

    print('pick completo')

    neurons = total_classes - len(hidden_classes)
    print(f'{X_train.shape[1]} {neurons}')
    model = contrastive_ae(input_neurons=X_train.shape[1],neurons=neurons)
    model.to(device)

    learning_rate = 0.0001
    opt = optim.Adam(model.parameters(),lr=learning_rate)
    margin = 10 # Parametro de afastamento das classes
    cae_lambda_1 = 0.1 # Parametro de influencia na distancia das classes
    
    
    for epoch in range(250):
        running_loss = 0
        running_ael = 0
        running_cl = 0
        contador = 0
        for data in train_loader:
            inputs, labels = data
            inputs = inputs.to(device)
            labels = labels.to(device)
            ps = pick_samples[contador]
            pl = pick_labels[contador]
            ps = ps.to(device)
            pl = pl.to(device)
            contador += 1
            #print(f'{contador}/{len(train_loader)}')
            opt.zero_grad()
            outputs, encoded = model(inputs)
            _, pick_encoded = model(ps)
            tam_pe = pick_encoded.shape[0]
            enc_total = torch.cat((encoded,pick_encoded), dim=0)
            enc_total_norm = normalize(enc_total)
            encoded_norm = enc_total_norm[:-tam_pe]
            pick_encoded_norm = enc_total_norm[-tam_pe:]
            loss, ael, cl = AEContrastive(input=inputs, output=outputs, embeddings = encoded_norm, labels=labels, pick_encoded = pick_encoded_norm, pick_labels = pl, margin=margin, lambda1=cae_lambda_1)
            #loss = dummyMSELoss(inputs,outputs)
            loss.backward()
            opt.step()
            running_loss += loss.item()
            running_ael += ael.item()
            running_cl += cl.item()
            #print(f'loss = {loss.item()} batch_size = {size}')
        print(f'filename: {filename} Epoch {epoch+1} loss:{running_loss/len(train_loader)} ael:{running_ael/len(train_loader)} cl:{running_cl/len(train_loader)}')

    model_e = encoder(model=model)
    model_e.to(device)

    model_e.eval()

    X_train = torch.tensor(np.array(X_train), dtype=torch.float)
    y_train = torch.tensor(np.array(y_train), dtype=torch.float)

    train_encoded = model_e(X_train.to(device))

    train_encoded = train_encoded.detach().cpu().numpy()

    train_encoded_df = pd.DataFrame(train_encoded)
    train_encoded_df['Label'] = y_train.detach().cpu().numpy().astype(int)
    display(train_encoded_df)

    score = davies_bouldin_score(train_encoded_df.drop(columns=['Label']), train_encoded_df['Label'])
    print("Davies-Bouldin Score:", score)

    X_val = torch.tensor(np.array(X_val), dtype=torch.float)

    val_encoded = model_e(X_val.to(device))

    val_encoded = val_encoded.detach().cpu().numpy()

    val_encoded_df = pd.DataFrame(val_encoded)

    val_encoded_df['Label'] = y_val

    display(val_encoded_df)

    X_test = torch.tensor(np.array(X_test), dtype=torch.float)

    test_encoded = model_e(X_test.to(device))

    test_encoded = test_encoded.detach().cpu().numpy()

    test_encoded_df = pd.DataFrame(test_encoded)

    test_encoded_df['Label'] = y_test

    display(test_encoded_df)

    train_encoded_df.to_csv(f'train_encoded_CADE_cod_pick_class_squared_normalized_{filename}.csv',index=False)
    val_encoded_df.to_csv(f'val_encoded_CADE_cod_pick_class_squared_normalized_{filename}.csv',index=False)
    test_encoded_df.to_csv(f'test_encoded_CADE_cod_pick_class_squared_normalized_{filename}.csv',index=False)

5_hidden


cuda


[0.0, 1.0, 2.0, 3.0, 4.0, 6.0]


pick completo
82 6


filename: 5_hidden Epoch 1 loss:3.704752619621374 ael:0.02503904212907986 cl:36.79713515122613


...

/tmp/ipykernel_887182/3449680296.py:91: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  X_train = torch.tensor(np.array(X_train), dtype=torch.float)
/tmp/ipykernel_887182/3449680296.py:92: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  y_train = torch.tensor(np.array(y_train), dtype=torch.float)


,0,1,2,3,4,5,Label
0,-0.040691,-0.032932,-0.037882,-0.051103,-0.024731,-0.024219,1
1,-0.028289,-0.042353,-0.042344,-0.035388,-0.046168,-0.034631,0
2,-0.051200,-0.030225,-0.045836,-0.028097,-0.031909,-0.061703,2
3,-0.034350,-0.027865,-0.035685,-0.044547,-0.018331,-0.023210,1
4,-0.012465,-0.022401,-0.023921,-0.011339,-0.017951,-0.028282,4
...,...,...,...,...,...,...,...
2053285,-0.028301,-0.042359,-0.042351,-0.035405,-0.046173,-0.034630,0
2053286,-0.051205,-0.030228,-0.045841,-0.028098,-0.031914,-0.061707,2
2053287,-0.051209,-0.030230,-0.045845,-0.028099,-0.031917,-0.061710,2
2053288,-0.028283,-0.042350,-0.042341,-0.035378,-0.046165,-0.034631,0


Davies-Bouldin Score: 0.5761587876070596


,0,1,2,3,4,5,Label
0,-0.029239,-0.043300,-0.041827,-0.035562,-0.047909,-0.035280,0
1,-0.051187,-0.030215,-0.045821,-0.028092,-0.031895,-0.061693,2
2,-0.039754,-0.032127,-0.039756,-0.054188,-0.024070,-0.023774,1
3,-0.053348,-0.039008,-0.002470,-0.034627,-0.045890,-0.036343,3
4,-0.039762,-0.031931,-0.038041,-0.052815,-0.024251,-0.024112,1
...,...,...,...,...,...,...,...
519951,-0.040344,-0.032203,-0.041154,-0.056125,-0.023594,-0.023259,1
519952,-0.011878,-0.020736,-0.021588,-0.009574,-0.015656,-0.027774,4
519953,-0.028298,-0.042357,-0.042349,-0.035401,-0.046172,-0.034630,0
519954,-0.038885,-0.032603,-0.038591,-0.048861,-0.024001,-0.023705,1


,0,1,2,3,4,5,Label
0,-0.028290,-0.042354,-0.042345,-0.035389,-0.046168,-0.034631,0
1,-0.029595,-0.043577,-0.042220,-0.036191,-0.048219,-0.035290,0
2,-0.039301,-0.032382,-0.038830,-0.052636,-0.024991,-0.024381,1
3,-0.046672,-0.030900,-0.045177,-0.067724,-0.018224,-0.019842,1
4,-0.028782,-0.042655,-0.041153,-0.034787,-0.047007,-0.035082,0
...,...,...,...,...,...,...,...
649942,-0.039611,-0.031827,-0.037831,-0.052503,-0.024159,-0.024131,1
649943,-0.028669,-0.042659,-0.041099,-0.034591,-0.047065,-0.035134,0
649944,-0.028718,-0.042648,-0.041127,-0.034686,-0.047021,-0.035103,0
649945,-0.052678,-0.038841,-0.002546,-0.033938,-0.045726,-0.036377,3
